# A3 Window C — OSM-anchored seat-placement prototype (re-run notebook)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Starive/A3-for-ARCH7476/blob/main/a3_seat_C.ipynb)

**How to run:** open this file in Google Colab or Jupyter and choose **Runtime → Run all**. Nothing else needs to be uploaded: the notebook carries its own copy of the code and the reference results, and it needs no internet data download (the OSM node coordinates are stored in the code with their source links).

**What it does:** rebuilds the inputs, runs all 36 scenarios (6 seat variants × 3 corridor half-widths × 2 flow-edge proxies), redraws the figures, and checks the new results against the reference results.

**Limits:** the garden location comes from OpenStreetMap (way 27059126). Window C, the flow edges, the corridor widths and the comfort thresholds are researcher assumptions. The results are a geometric test, not evidence of pedestrian flow or comfort. See `README.md`.

© OpenStreetMap contributors (ODbL) for the node coordinates.

## 1. Environment

In [ ]:
import importlib, subprocess, sys
need = ['geopandas', 'shapely', 'pyproj', 'pandas', 'matplotlib', 'numpy', 'PIL']
pip_name = {'PIL': 'pillow'}
missing = [m for m in need if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + [pip_name.get(m, m) for m in missing], check=True)
import geopandas, shapely, pyproj, pandas, matplotlib, platform
print('python', platform.python_version())
for m in (geopandas, shapely, pyproj, pandas, matplotlib):
    print(m.__name__, m.__version__)
print('Reference run used: python 3.13.5, geopandas 1.1.2, shapely 2.1.2, pyproj 3.7.2, pandas 2.2.3, matplotlib 3.10.8')

## 2. Code and reference results
If the notebook sits inside the extracted package, the package files are used. Otherwise the embedded copies below are written to a new folder `A3_C_OSM_Prototype_rerun/`.

In [ ]:
from pathlib import Path
MAIN_PY = '"""A3 Window C reproducible, OSM-anchored *exploratory* prototype.\n\nActual OSM: park northeast/east-boundary nodes and node links. All Window C\nplacement parameters, edge-as-flow proxies, thresholds are researcher assumptions.\nThese outputs do not establish observed walking activity or actual comfort.\n"""\nfrom pathlib import Path\nimport json, sys, platform\nimport numpy as np, pandas as pd, geopandas as gpd\nfrom pyproj import Transformer\nfrom shapely.geometry import Point, Polygon, LineString\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Polygon as MPolygon, Patch\nfrom matplotlib.lines import Line2D\nROOT=Path(__file__).resolve().parent.parent\nIN=ROOT/\'inputs\'; OUT=ROOT/\'outputs\'\nIN.mkdir(exist_ok=True,parents=True); OUT.mkdir(exist_ok=True,parents=True)\nCONFIG={\n \'crs\':\'EPSG:2326\', \'window_size_m\':6.0,\n \'c_window_start_south_of_osm_ne_m\':5.0,\n \'seat_length_m\':2.0, \'seat_width_m\':1.0,\n \'positions_m\':[2.2,3.2,4.3],\n \'w_values_m\':[.75,1.0,1.5], \'base_w_m\':1.0,\n \'c1_m\':.30, \'c2_m\':1.50, \'band_threshold_s_pct\':60.0,\n \'flow_proxy_edges\':[\'east_garden_boundary\',\'north_assumed_window_edge\'],\n \'source\':\'OpenStreetMap (OSM) way 27059126; tagged garden perimeter nodes; accessed 2026-11-13\',\n \'warning\':\'EAST EDGE IS A GARDEN BOUNDARY, NOT VERIFIED FOOTWAY; NORTH EDGE CONSTRUCTED. NO MEASURED WALKING FLOW.\'\n}\n# The following are read from individual public OpenStreetMap node pages;\n# IDs and coordinates retained for traceability. The northwestern node remains unverified.\nNODES=[\n (\'8308187127\',\'W boundary\',22.2803521,114.1750059),\n (\'296894101\',\'SW\',22.2801692,114.1750221),\n (\'12270766150\',\'S-mid\',22.2802022,114.1754572),\n (\'296894102\',\'SE\',22.2802402,114.1759591),\n (\'296894103\',\'NE\',22.2806549,114.1759247),\n]\nproj=Transformer.from_crs(\'EPSG:4326\',\'EPSG:2326\',always_xy=True)\nnode_data=[]\nfor osm_id,label,lat,lon in NODES:\n x,y=proj.transform(lon,lat)\n node_data.append({\'osm_id\':osm_id,\'label\':label,\'lat\':lat,\'lon\':lon,\n       \'url\':f\'https://www.openstreetmap.org/node/{osm_id}\',\'geometry\':Point(x,y)})\nnode_gdf=gpd.GeoDataFrame(node_data,crs=CONFIG[\'crs\'])\nnode_gdf.to_file(IN/\'osm_garden_nodes_verified.geojson\',driver=\'GeoJSON\')\nNE=np.array(node_gdf.loc[node_gdf.label==\'NE\',\'geometry\'].iloc[0].coords[0])\nSE=np.array(node_gdf.loc[node_gdf.label==\'SE\',\'geometry\'].iloc[0].coords[0])\nt=(SE-NE)/np.linalg.norm(SE-NE)  # unit vector along east garden boundary toward south\nn=np.array([t[1],-t[0]]) # inward (west) normal\nassert n[0]<0\nO=NE+CONFIG[\'c_window_start_south_of_osm_ne_m\']*t\n\ndef xy(u,v): return tuple(O+float(u)*n+float(v)*t)\ndef local_poly(coords):return Polygon([xy(u,v) for u,v in coords])\nW=CONFIG[\'window_size_m\']\nwindow=local_poly([(0,0),(W,0),(W,W),(0,W)])\nedge_east=LineString([tuple(NE),tuple(SE)])\nedge_north=LineString([xy(0,0),xy(W,0)])\nwindow_gdf=gpd.GeoDataFrame([{\'id\':\'C\',\'source\':\'RESEARCHER-DEFINED 6x6m, OSM-east-boundary-anchored\',\'geometry\':window}],crs=CONFIG[\'crs\'])\nwindow_gdf.to_file(IN/\'window_C_osm_anchored_ASSUMED.geojson\',driver=\'GeoJSON\')\nedge_gdf=gpd.GeoDataFrame([\n {\'name\':\'east_garden_boundary\',\'source\':\'OSM WAY 27059126 - garden boundary, pedestrian flow UNVERIFIED\',\'geometry\':edge_east},\n {\'name\':\'north_assumed_window_edge\',\'source\':\'RESEARCHER-CONSTRUCTED - not a mapped path or observed flow\',\'geometry\':edge_north},\n],crs=CONFIG[\'crs\'])\nedge_gdf.to_file(IN/\'flow_edge_proxies_NOT_PATHS.geojson\',driver=\'GeoJSON\')\nrows=[]\nfor i,location in enumerate([\'Near\',\'Centre\',\'Far\']):\n for orientation in [\'parallel\',\'perpendicular\']:\n  vid=f\'S{2*i+(1 if orientation=="parallel" else 2)}\'\n  rows.append({\'variant\':vid,\'position\':location,\'distance_from_flow_edge_m\':CONFIG[\'positions_m\'][i],\n               \'orientation\':orientation,\'seat_L_m\':2.0,\'seat_W_m\':1.0,\'value_status\':\'RESEARCHER ASSUMPTION\'})\nvariants=pd.DataFrame(rows)\nvariants.to_csv(IN/\'variants.csv\',index=False)\n\ndef seat_polygon(distance,ori,edge):\n if edge==\'east_garden_boundary\':\n  uc,vc=distance,3.0\n  du,dv=(.5,1.0) if ori==\'parallel\' else (1.0,.5)\n else:\n  uc,vc=3.0,distance\n  du,dv=(1.0,.5) if ori==\'parallel\' else (.5,1.0)\n return local_poly([(uc-du,vc-dv),(uc+du,vc-dv),(uc+du,vc+dv),(uc-du,vc+dv)])\n\nallresults=[]\nfor edge in CONFIG[\'flow_proxy_edges\']:\n line=edge_east if edge==\'east_garden_boundary\' else edge_north\n for w in CONFIG[\'w_values_m\']:\n  corridor=line.buffer(w,cap_style=\'flat\')\n  comfort=line.buffer(w+CONFIG[\'c2_m\'],cap_style=\'flat\').difference(\n    line.buffer(w+CONFIG[\'c1_m\'],cap_style=\'flat\')).intersection(window)\n  for v in rows:\n   seat=seat_polygon(v[\'distance_from_flow_edge_m\'],v[\'orientation\'],edge)\n   ov=seat.intersection(corridor).area\n   band=seat.intersection(comfort).area\n   status=\'Conflict\' if ov>1e-8 else \'Clear under assumed geometry\'\n   allresults.append({\'variant\':v[\'variant\'],\'position\':v[\'position\'],\'orientation\':v[\'orientation\'],\n     \'flow_proxy\':edge,\'w_m\':w,\'corridor_overlap_m2\':round(ov,5),\n     \'comfort_band_share_pct\':round(band*100/seat.area,3),\'seat_area_m2\':round(seat.area,5),\n     \'inside_window\':window.covers(seat),\'geom_status\':status,\n     \'band_threshold_pass\':bool(ov < 1e-8 and 100*band/seat.area >= CONFIG[\'band_threshold_s_pct\']),\n     \'data_status\':\'OSM-ANCHORED LOCATION; WINDOW, FLOW, THRESHOLD ASSUMED\'})\nres=pd.DataFrame(allresults)\nassert len(res)==36 and res.inside_window.all() and np.allclose(res.seat_area_m2,2.0)\nres.to_csv(OUT/\'a3_results_osm_anchored.csv\',index=False)\nbase=res[(res.flow_proxy==\'east_garden_boundary\')&(res.w_m==1.)].copy().sort_values(\'variant\')\nbase.to_csv(OUT/\'a3_base_comparison.csv\',index=False)\nknown=base.loc[base.variant==\'S1\',\'comfort_band_share_pct\'].iloc[0]\nassert np.isclose(known,80.0)\n# source layer coordinates checks\nedgelen=edge_east.length\nassert 40 < edgelen < 55\nchecks=pd.DataFrame([\n (\'East OSM boundary length\',\'40 to 55m\',f\'{edgelen:.2f} m\',\'PASS, based on cited OSM node coords\'),\n (\'36 scenario output rows\',\'36\',str(len(res)),\'PASS\'),\n (\'Seat area after rotation\',\'2.00 m2\',f"{res.seat_area_m2.min():.2f} to {res.seat_area_m2.max():.2f} m2",\'PASS\'),\n (\'Known case S1 band share east/w=1\',\'80.0% by hand\',\'%.1f%%\'%known,\'PASS\'),\n (\'All seats in assumed Window C\',\'True\',str(bool(res.inside_window.all())),\'PASS\'),\n (\'Footway/observed flow at NE garden edge\',\'OSM footway/field verification\',\'OSM garden boundary only\',\'PENDING\'),\n (\'Window C true extent\',\'surveyed C polygon\',\'6x6m assumption\',\'PENDING\'),\n (\'Comfort threshold validation\',\'observational/literature evidence\',\'c1/c2/s assumed\',\'PENDING\'),\n],columns=[\'check\',\'expected\',\'observed\',\'status\'])\nchecks.to_csv(OUT/\'a3_verification_log.csv\',index=False)\n\nplt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':10})\nGREEN=\'#236456\'; SAGE=\'#b7d0c7\'; MUSTARD=\'#c6ac67\'; ORANGE=\'#ac6042\'; DARK=\'#283b38\'\n# site footprint figure based on OSM nodes; missing NW not connected\nfig,ax=plt.subplots(figsize=(9,5.4))\nnode_order=[\'W boundary\',\'SW\',\'S-mid\',\'SE\',\'NE\']\npolypts=[node_gdf.loc[node_gdf.label==name,\'geometry\'].iloc[0].coords[0] for name in node_order]\nax.plot([p[0] for p in polypts],[p[1] for p in polypts],color=GREEN,lw=2.5,label=\'Verified garden perimeter segments\')\nfor row in node_gdf.itertuples():\n ax.scatter(row.geometry.x,row.geometry.y,s=30,color=GREEN)\n ax.text(row.geometry.x+1,row.geometry.y+1,row.label,fontsize=9)\nwx,wy=window.exterior.xy\nax.fill(wx,wy,color=MUSTARD,alpha=.50,ec=ORANGE,lw=2,label=\'Window C (6×6m assumed)\')\nax.plot(*edge_east.xy,color=ORANGE,linewidth=4,label=\'OSM garden east edge (not footway)\')\nax.annotate(\'Window C\',xy=window.centroid.coords[0],xytext=(NE[0]-40,NE[1]-3),arrowprops=dict(arrowstyle=\'->\',color=ORANGE),fontsize=11,color=ORANGE)\nax.set_aspect(\'equal\');ax.set_xlabel(\'Hong Kong Grid Easting (m)\');ax.set_ylabel(\'Northing (m)\')\nax.set_title(\'OSM-sourced garden geometry + researcher-defined Window C\',weight=\'bold\')\nax.legend(fontsize=8,loc=\'lower left\',frameon=False)\nax.grid(alpha=.12); fig.text(.02,.008,\'OSM way 27059126 | 5 individually sourced nodes; NW node not retrieved | C size and flow unverified\',color=ORANGE,fontsize=8)\nfig.tight_layout(rect=[0,.035,1,1]);fig.savefig(OUT/\'osm_context.png\',dpi=180);plt.close(fig)\n# plan in local window coords (east edge u=0; inward u positive)\nfig,ax=plt.subplots(figsize=(8.9,6))\nax.add_patch(MPolygon(np.array([(0,0),(W,0),(W,W),(0,W)]),closed=True,facecolor=\'#eff5f0\',edgecolor=GREEN,lw=2))\nw=1.;c1=CONFIG[\'c1_m\'];c2=CONFIG[\'c2_m\']\nax.axvspan(0,w,color=SAGE,alpha=.80,label=\'Route corridor *proxy*\')\nax.axvspan(w+c1,w+c2,color=MUSTARD,alpha=.48,label=\'Assumed comfort band\')\nfor bound,color,ls in [(w,GREEN,\'--\'),(w+c1,ORANGE,\':\'),(w+c2,ORANGE,\':\')]:ax.axvline(bound,color=color,lw=1.5,ls=ls)\nfor v in rows:\n seat=seat_polygon(v[\'distance_from_flow_edge_m\'],v[\'orientation\'],\'east_garden_boundary\')\n # convert points back to local (u,v) for plotting\n uv=[(float(np.dot(np.array(p)-O,n)),float(np.dot(np.array(p)-O,t))) for p in seat.exterior.coords]\n chosen=v[\'variant\']==\'S1\'\n ax.add_patch(MPolygon(np.array(uv),closed=True,facecolor=GREEN if chosen else \'white\',edgecolor=GREEN if chosen else ORANGE,lw=1.7,alpha=.92))\n ucenter=v[\'distance_from_flow_edge_m\']; vcenter=3\n ax.text(ucenter,vcenter,v[\'variant\'],fontsize=10,color=\'white\' if chosen else DARK,ha=\'center\',va=\'center\',fontweight=\'bold\')\nax.text(.10,5.75,\'OSM EAST GARDEN BOUNDARY\',color=GREEN,fontsize=9,weight=\'bold\',rotation=90,va=\'top\')\nax.set_aspect(\'equal\');ax.set_xlim(-.3,6.35);ax.set_ylim(-.35,6.45)\nax.set_xlabel(\'Distance inward from east garden edge (m)\');ax.set_ylabel(\'Distance south along edge (m)\')\nax.set_title(\'Window C  |  6 assumed seat placements\',fontweight=\'bold\')\nax.legend(loc=\'upper right\',frameon=False,fontsize=9)\nfig.text(.02,.009,\'OSM-anchored, NOT observed pedestrian flow; Window C = researcher-defined 6×6m; thresholds provisional\',color=ORANGE,fontsize=8)\nfig.tight_layout(rect=[0,.035,1,1]);fig.savefig(OUT/\'a3_plan_osm.png\',dpi=180);plt.close(fig)\n# decision chart\nfig,ax=plt.subplots(figsize=(9.5,4.4))\ncolors=[GREEN if r.band_threshold_pass else \'#b9c4bd\' for r in base.itertuples()]\nax.bar(base.variant,base.comfort_band_share_pct,color=colors,edgecolor=GREEN,width=.65)\nax.axhline(CONFIG[\'band_threshold_s_pct\'],color=ORANGE,lw=2,ls=\'--\')\nax.text(5.55,CONFIG[\'band_threshold_s_pct\']+2,\'s = 60% (ASSUMED)\',color=ORANGE,fontsize=9,ha=\'right\')\nfor i,r in enumerate(base.itertuples()):ax.text(i,r.comfort_band_share_pct+2,f\'{r.comfort_band_share_pct:.0f}%\',ha=\'center\',fontsize=10)\nax.set_ylim(0,110);ax.set_ylabel(\'Comfort-band share (%)\');ax.set_title(\'Conditional comparison | east garden-edge proxy, w = 1.0 m\',fontweight=\'bold\')\nax.spines[[\'top\',\'right\']].set_visible(False)\nfig.text(.02,.008,\'OSM-ANCHORED EXPLORATION • C window / edge-flow / c1 c2 s remain assumed; no behavioural conclusion\',color=ORANGE,fontsize=8)\nfig.tight_layout(rect=[0,.035,1,1]);fig.savefig(OUT/\'a3_threshold_chart.png\',dpi=180);plt.close(fig)\n# flip chart\nfig,ax=plt.subplots(figsize=(9.5,4.4))\nfor v in [\'S1\',\'S2\',\'S3\',\'S4\',\'S5\',\'S6\']:\n part=res[(res.variant==v)&(res.flow_proxy==\'east_garden_boundary\')].sort_values(\'w_m\')\n ax.plot(part.w_m,part.comfort_band_share_pct,marker=\'o\',label=v,alpha=1 if v==\'S1\' else .65,lw=2.5 if v==\'S1\' else 1.25)\nax.axhline(CONFIG[\'band_threshold_s_pct\'],color=ORANGE,lw=2,ls=\'--\',label=\'s = 60% assumed\')\nax.set_xticks(CONFIG[\'w_values_m\']);ax.set_ylim(-3,105);ax.set_xlim(.7,1.55)\nax.set_xlabel(\'Corridor half-width w (m; assumed)\');ax.set_ylabel(\'Comfort-band share (%)\')\nax.set_title(\'Threshold sensitivity | east garden-edge proxy\',fontweight=\'bold\')\nax.grid(alpha=.15);ax.legend(ncol=4,frameon=False,fontsize=9)\nfig.text(.02,.008,\'S1 crosses s=60% between tested w=.75 and w=1.0 • scenario dependence, not measured comfort\',color=ORANGE,fontsize=8)\nfig.tight_layout(rect=[0,.035,1,1]);fig.savefig(OUT/\'a3_flip_chart.png\',dpi=180);plt.close(fig)\nsummary={\'status\':\'OSM-ANCHORED GEOMETRY / ALL FLOW AND BEHAVIOR ASSUMPTIONS UNVALIDATED\',\n \'osm_garden_way\':\'https://www.openstreetmap.org/way/27059126\',\'osm_ne_node\':\'296894103\',\n \'osm_se_node\':\'296894102\',\'osm_east_boundary_length_m\':round(edgelen,2),\n \'window_C\':\'researcher-defined 6x6m patch along east garden edge 5m south of OSM NE node\',\n \'base_east_proxy_w1_band_share_pct\':dict(zip(base.variant,base.comfort_band_share_pct)),\n \'base_best_conditional\':base.sort_values(\'comfort_band_share_pct\',ascending=False).iloc[0].variant,\n \'source_caveat\':\'OSM east garden boundary is not tagged as footway. North flow edge is an assumed window-side segment.\',\n \'assumption_note\':\'c1=.3m, c2=1.5m, s=60% researcher-defined; no measured pedestrian routes or stop counts.\'}\n(OUT/\'a3_summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf8\')\n(IN/\'osm_provenance.json\').write_text(json.dumps({\'access_date\':\'2026-11-13\',\'garden_way\':\'https://www.openstreetmap.org/way/27059126\',\'nodes\':[{k:v for k,v in item.items() if k!=\'geometry\'} for item in node_data],\'status\':\'Only 5 of 6 unique garden boundary nodes retrieved as verifiable HTML; northeast and east side complete.\'},indent=2),encoding=\'utf8\')\n(OUT/\'versions.txt\').write_text(\'\\n\'.join([\'python \'+platform.python_version(),\'geopandas \'+gpd.__version__,\'shapely \'+__import__(\'shapely\').__version__,\'matplotlib \'+matplotlib.__version__,\'pyproj \'+__import__(\'pyproj\').__version__,\'pandas \'+pd.__version__]),encoding=\'utf8\')\nprint(json.dumps(summary,indent=2))\n'
PANELS_PY = "from PIL import Image, ImageDraw, ImageFont\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/'outputs'\nW,H=1800,1120\ncan=Image.new('RGB',(W,H),'white');D=ImageDraw.Draw(can)\ntry:\n title=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',32)\n normal=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',21)\n small=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',18)\nexcept: title=normal=small=ImageFont.load_default()\nD.text((32,10),'S1–S6: alternate layouts in the same 6 × 6 m Window C',font=title,fill='#244e45')\nD.text((32,59),'Each panel shows one separate 2 × 1 m seat. OSM east boundary = location anchor; pedestrian flow remains unverified.',font=normal,fill='#825f3e')\ncolors={'corridor':'#b7cec6','comfort':'#e2d2a1','seat':'#236456','edge':'#ae6953'}\nmarginX=40; top=116; panelW=565; panelH=470; padX=50;padY=44\nfor i,(name,dist,parallel) in enumerate([('S1',2.2,True),('S2',2.2,False),('S3',3.2,True),('S4',3.2,False),('S5',4.3,True),('S6',4.3,False)]):\n col=i%3;row=i//3;x0=marginX+col*590;y0=top+row*480\n D.rounded_rectangle((x0,y0,x0+panelW,y0+panelH),radius=14,fill='#f6f9f7',outline='#bbcbc3',width=2)\n D.text((x0+22,y0+12),f'{name}  ·  '+['Near','Near','Centre','Centre','Far','Far'][i]+'  ·  '+('parallel' if parallel else 'perpendicular'),font=normal,fill='#244e45')\n edgeX=x0+72; edgeY=y0+76;sz=51\n # C at map coordinate u 0..6, v 0..6, u to right (inward), v down\n D.rectangle((edgeX,edgeY,edgeX+6*sz,edgeY+6*sz),fill='#eff5ef',outline='#236456',width=3)\n D.rectangle((edgeX,edgeY,edgeX+1*sz,edgeY+6*sz),fill=colors['corridor'])\n D.rectangle((edgeX+1.3*sz,edgeY,edgeX+2.5*sz,edgeY+6*sz),fill=colors['comfort'])\n for u in [1,1.3,2.5]:\n  xx=round(edgeX+u*sz)\n  D.line((xx,edgeY,xx,edgeY+6*sz),fill='#ae6953' if u!=1 else '#266454',width=2)\n uc=dist; vc=3.0\n if parallel: u_rad=.5;v_rad=1.0\n else: u_rad=1.0;v_rad=.5\n rect=(round(edgeX+(uc-u_rad)*sz),round(edgeY+(vc-v_rad)*sz),round(edgeX+(uc+u_rad)*sz),round(edgeY+(vc+v_rad)*sz))\n D.rectangle(rect,fill=colors['seat'] if i==0 else '#fffefa',outline='#236456',width=3)\n labelB=D.textbbox((0,0),name,font=normal)\n D.text(((rect[0]+rect[2]-(labelB[2]-labelB[0]))/2,(rect[1]+rect[3]-(labelB[3]-labelB[1]))/2),name,font=normal,fill='white' if i==0 else '#26493f')\n D.text((edgeX,edgeY+6*sz+7),'0 m',font=small,fill='#5a706a')\n D.text((edgeX+6*sz-40,edgeY+6*sz+7),'6 m',font=small,fill='#5a706a')\n D.text((x0+22,y0+panelH-31),'Sourced boundary geometry · assumed seats and bands',font=small,fill='#a26443')\nD.text((36,H-32),'EXPLORATORY GEOMETRIC PROXY · NOT OBSERVED PEDESTRIAN FLOW · NOT VALIDATED COMFORT',font=small,fill='#a26443')\ncan.save(OUT/'a3_six_variant_plans.png')\nprint(OUT/'a3_six_variant_plans.png')\n"
FIGS_PY = '"""Report-style figures drawn from the results table.\nUsage: python code/make_figures.py   (run after code/a3_osm_anchor.py)\nNeeds only matplotlib. Reads outputs/a3_results_osm_anchored.csv and inputs/variants.csv.\nGeometry is drawn in the local Window C frame: east edge at x = 0, north edge at y = 0, metres."""\nimport csv\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Rectangle\n\nROOT = Path(__file__).resolve().parent.parent\nOUT = ROOT / "outputs"\nWINDOW, SEAT_L, SEAT_W = 6.0, 2.0, 1.0\nC1, C2, S_LINE = 0.30, 1.50, 60.0          # must match CONFIG in a3_osm_anchor.py\nEDGE, EDGE_LABEL = "east_garden_boundary", "east edge proxy"\nGREEN, TEAL, TAN, ORANGE, RULE = "#1f5a4e", "#9fbfb0", "#dcc3a5", "#c8623f", "#d9532b"\n\ndef read(path):\n    with open(path, encoding="utf8", newline="") as f:\n        return list(csv.DictReader(f))\n\nrows = [r for r in read(OUT / "a3_results_osm_anchored.csv") if r["flow_proxy"] == EDGE]\ndist = {r["variant"]: float(r["distance_from_flow_edge_m"]) for r in read(ROOT / "inputs" / "variants.csv")}\nwidths = sorted({float(r["w_m"]) for r in rows})\nids = sorted({r["variant"] for r in rows})\nget = lambda vid, w: next(r for r in rows if r["variant"] == vid and abs(float(r["w_m"]) - w) < 1e-9)\nBASE_W = 1.0 if 1.0 in widths else widths[len(widths) // 2]\n\n# Figure 1: six alternative placements of one seat\nfig, axes = plt.subplots(2, 3, figsize=(13.6, 9.4))\nfor ax, vid in zip(axes.flat, ids):\n    r = get(vid, BASE_W)\n    parallel = r["orientation"] == "parallel"\n    du, dv = (SEAT_W / 2, SEAT_L / 2) if parallel else (SEAT_L / 2, SEAT_W / 2)\n    d = dist[vid]\n    ax.add_patch(Rectangle((-WINDOW, -WINDOW), WINDOW, WINDOW, fill=False, ec="#4b6b62", ls="--", lw=1))\n    ax.add_patch(Rectangle((-BASE_W, -WINDOW), BASE_W, WINDOW, fc=TAN, ec="none", alpha=0.65))\n    ax.add_patch(Rectangle((-(BASE_W + C2), -WINDOW), C2 - C1, WINDOW, fc=TEAL, ec="none", alpha=0.5))\n    ax.plot([0, 0], [0, -WINDOW], color=GREEN, lw=3)\n    conflict = float(r["corridor_overlap_m2"]) > 1e-8\n    ax.add_patch(Rectangle((-d - du, -WINDOW / 2 - dv), 2 * du, 2 * dv, fc=ORANGE, ec="#8f1d1d" if conflict else ORANGE,\n                           lw=2.5 if conflict else 1.5, alpha=0.7))\n    ax.set_title(f\'{vid} · {r["position"].lower()} / {r["orientation"]}\\n\'\n                 f\'Band share {float(r["comfort_band_share_pct"]):.0f}% · overlap {float(r["corridor_overlap_m2"]):.2f} m²\',\n                 fontsize=11, color="#1f3b34")\n    ax.set_xlim(-WINDOW - 0.5, 0.45); ax.set_ylim(-WINDOW - 0.5, 0.4); ax.set_aspect("equal")\n    ax.set_xlabel("Local x (m) · east edge at x=0", fontsize=8.5)\n    ax.set_ylabel("Local y (m) · north edge at y=0", fontsize=8.5)\n    ax.grid(alpha=0.12)\nfig.suptitle(f"Window C / six ALTERNATIVE placements of ONE 2 × 1 m seat\\n"\n             f"Assumed 6 × 6 m geometry; {EDGE_LABEL}, w = {BASE_W:.1f} m", fontsize=15, fontweight="bold", color=GREEN)\nfig.text(0.5, 0.015, "Schematic geometry only • green = proposed flow-adjacent band • tan = assumed corridor • orange = seat",\n         ha="center", fontsize=9.5, color="#5f786f")\nfig.tight_layout(rect=(0, 0.03, 1, 0.93), h_pad=2.2)\nfig.savefig(OUT / "fig_six_placements.png", dpi=160); plt.close(fig)\n\n# Figure 2: base-case comparison with the threshold drawn in\nbase = [get(v, BASE_W) for v in ids]\nshares = [float(r["comfort_band_share_pct"]) for r in base]\npassed = [r["band_threshold_pass"].strip().lower() == "true" for r in base]\nlead = ids[max(range(len(ids)), key=lambda i: shares[i])]\nfig, ax = plt.subplots(figsize=(10, 4.9))\nax.bar(ids, shares, color=[GREEN if p else "#5f7d73" for p in passed], width=0.66)\nfor i, s in enumerate(shares):\n    ax.text(i, s + 1.5, f"{s:.0f}%", ha="center", fontsize=10, color="#2a3d38")\nax.axhline(S_LINE, color=RULE, lw=2, ls="--", label=f"Assumed research threshold s = {S_LINE:.0f}%")\nax.set_ylim(0, 105); ax.set_ylabel("Proposed flow-adjacent band share (%)")\nax.set_title(f"Base-case {lead} leads; proxy threshold is NOT a comfort standard", fontweight="bold", color=GREEN)\nax.legend(frameon=False, loc="upper right", fontsize=9)\nax.spines[["top", "right"]].set_visible(False)\nfig.text(0.5, 0.02, f"East boundary proxy · corridor half-width w = {BASE_W:.2f} m · c₁={C1:.2f}m, c₂={C2:.2f}m",\n         ha="center", fontsize=9, color="#5f786f")\nfig.tight_layout(rect=(0, 0.05, 1, 1))\nfig.savefig(OUT / "fig_base_comparison.png", dpi=160); plt.close(fig)\n\n# Figure 3: band share across corridor widths, with corridor conflicts marked\nfig, ax = plt.subplots(figsize=(10, 4.9))\nfor vid in ids:\n    pts = [get(vid, w) for w in widths]\n    ys = [float(r["comfort_band_share_pct"]) for r in pts]\n    is_lead = vid == lead\n    ax.plot(widths, ys, marker="o", color=GREEN if is_lead else "#8aa59c", lw=2.8 if is_lead else 1.4, zorder=3 if is_lead else 2)\n    for w, y, r in zip(widths, ys, pts):\n        if float(r["corridor_overlap_m2"]) > 1e-8:\n            ax.plot(w, y, marker="X", ms=11, color="#8f1d1d", zorder=4)\n    ax.text(widths[-1] + 0.02, ys[-1], vid, va="center", fontsize=9.5, fontweight="bold" if is_lead else "normal", color="#2a3d38")\nax.axhline(S_LINE, color=RULE, lw=2, ls="--", label=f"Assumed research threshold s = {S_LINE:.0f}%")\nax.plot([], [], marker="X", ls="none", ms=10, color="#8f1d1d", label="Seat overlaps the corridor")\nlead_vals = [float(get(lead, w)["comfort_band_share_pct"]) for w in widths]\nverdict = "stays above the line at every tested width" if min(lead_vals) >= S_LINE else "drops below the line at one or more tested widths"\nax.set_title(f"{lead} {verdict} (min {min(lead_vals):.0f}%)", fontweight="bold", color=GREEN)\nax.set_xticks(widths); ax.set_xlim(widths[0] - 0.05, widths[-1] + 0.12); ax.set_ylim(-3, 105)\nax.set_xlabel("Corridor half-width w (m; assumed)"); ax.set_ylabel("Proposed flow-adjacent band share (%)")\nax.legend(frameon=False, fontsize=9, loc="upper left"); ax.grid(alpha=0.15)\nax.spines[["top", "right"]].set_visible(False)\nfig.tight_layout()\nfig.savefig(OUT / "fig_width_sensitivity.png", dpi=160); plt.close(fig)\nprint("Figures written to", OUT)\n'
CHECK_PY = '"""Compare a fresh run against the reference results shipped with the package.\nUsage: python code/check_reproduction.py   (run after code/a3_osm_anchor.py)\nUses only the Python standard library."""\nimport csv, sys\nfrom pathlib import Path\nROOT = Path(__file__).resolve().parent.parent\nTOL = 0.01  # m2 and percentage points\n\ndef load(path):\n    with open(path, encoding="utf8", newline="") as f:\n        return {(r["variant"], r["flow_proxy"], f\'{float(r["w_m"]):.2f}\'): r for r in csv.DictReader(f)}\n\ndef main(new_path=None, ref_path=None):\n    new = load(new_path or ROOT / "outputs" / "a3_results_osm_anchored.csv")\n    ref = load(ref_path or ROOT / "reference" / "a3_results_reference.csv")\n    problems = []\n    if set(new) != set(ref):\n        problems.append(f"scenario keys differ: {len(new)} new vs {len(ref)} reference")\n    for key in sorted(set(new) & set(ref)):\n        for col in ("corridor_overlap_m2", "comfort_band_share_pct"):\n            a, b = float(new[key][col]), float(ref[key][col])\n            if abs(a - b) > TOL:\n                problems.append(f"{key} {col}: got {a}, reference {b}")\n        if new[key]["band_threshold_pass"].strip().lower() != ref[key]["band_threshold_pass"].strip().lower():\n            problems.append(f"{key} band_threshold_pass differs")\n    if problems:\n        print("NOT REPRODUCED:")\n        for p in problems:\n            print("  -", p)\n        return 1\n    print(f"REPRODUCED: all {len(ref)} scenarios match the reference within {TOL}.")\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
REFERENCE_CSV = 'variant,flow_proxy,w_m,corridor_overlap_m2,comfort_band_share_pct,band_threshold_pass\nS1,east_garden_boundary,0.75,0.0,55.0,False\nS2,east_garden_boundary,0.75,0.0,52.5,False\nS3,east_garden_boundary,0.75,0.0,0.0,False\nS4,east_garden_boundary,0.75,0.0,2.5,False\nS5,east_garden_boundary,0.75,0.0,0.0,False\nS6,east_garden_boundary,0.75,0.0,0.0,False\nS1,east_garden_boundary,1.0,0.0,80.0,True\nS2,east_garden_boundary,1.0,0.0,60.0,True\nS3,east_garden_boundary,1.0,0.0,0.0,False\nS4,east_garden_boundary,1.0,0.0,15.0,False\nS5,east_garden_boundary,1.0,0.0,0.0,False\nS6,east_garden_boundary,1.0,0.0,0.0,False\nS1,east_garden_boundary,1.5,0.0,90.0,True\nS2,east_garden_boundary,1.5,0.3,60.0,False\nS3,east_garden_boundary,1.5,0.0,30.0,False\nS4,east_garden_boundary,1.5,0.0,40.0,False\nS5,east_garden_boundary,1.5,0.0,0.0,False\nS6,east_garden_boundary,1.5,0.0,0.0,False\nS1,north_assumed_window_edge,0.75,0.0,55.0,False\nS2,north_assumed_window_edge,0.75,0.0,52.5,False\nS3,north_assumed_window_edge,0.75,0.0,0.0,False\nS4,north_assumed_window_edge,0.75,0.0,2.5,False\nS5,north_assumed_window_edge,0.75,0.0,0.0,False\nS6,north_assumed_window_edge,0.75,0.0,0.0,False\nS1,north_assumed_window_edge,1.0,0.0,80.0,True\nS2,north_assumed_window_edge,1.0,0.0,60.0,True\nS3,north_assumed_window_edge,1.0,0.0,0.0,False\nS4,north_assumed_window_edge,1.0,0.0,15.0,False\nS5,north_assumed_window_edge,1.0,0.0,0.0,False\nS6,north_assumed_window_edge,1.0,0.0,0.0,False\nS1,north_assumed_window_edge,1.5,0.0,90.0,True\nS2,north_assumed_window_edge,1.5,0.3,60.0,False\nS3,north_assumed_window_edge,1.5,0.0,30.0,False\nS4,north_assumed_window_edge,1.5,0.0,40.0,False\nS5,north_assumed_window_edge,1.5,0.0,0.0,False\nS6,north_assumed_window_edge,1.5,0.0,0.0,False\n'
here = Path.cwd()
if (here / 'code' / 'a3_osm_anchor.py').exists():
    ROOT = here
    print('Using package files in', ROOT)
else:
    ROOT = here / 'A3_C_OSM_Prototype_rerun'
    (ROOT / 'code').mkdir(parents=True, exist_ok=True)
    (ROOT / 'code' / 'a3_osm_anchor.py').write_text(MAIN_PY, encoding='utf8')
    (ROOT / 'code' / 'make_variant_panels.py').write_text(PANELS_PY, encoding='utf8')
    (ROOT / 'code' / 'make_figures.py').write_text(FIGS_PY, encoding='utf8')
    print('Wrote embedded code to', ROOT)
(ROOT / 'code').mkdir(parents=True, exist_ok=True)
(ROOT / 'reference').mkdir(exist_ok=True)
if not (ROOT / 'code' / 'check_reproduction.py').exists():
    (ROOT / 'code' / 'check_reproduction.py').write_text(CHECK_PY, encoding='utf8')
if not (ROOT / 'reference' / 'a3_results_reference.csv').exists():
    (ROOT / 'reference' / 'a3_results_reference.csv').write_text(REFERENCE_CSV, encoding='utf8')

## 3. Run the workflow (36 scenarios, inputs, figures)

In [ ]:
import subprocess, sys
for script in ['a3_osm_anchor.py', 'make_variant_panels.py', 'make_figures.py']:
    r = subprocess.run([sys.executable, str(ROOT / 'code' / script)], cwd=str(ROOT), capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f'{script} failed; see the error above and the Troubleshooting section of README.md')

## 4. Reproduction check
Compares the fresh results with the reference results (tolerance 0.01). The expected message is **REPRODUCED**.

In [ ]:
r = subprocess.run([sys.executable, str(ROOT / 'code' / 'check_reproduction.py')], cwd=str(ROOT), capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'Results differ from the reference; see the list above'

## 5. Results

In [ ]:
import pandas as pd
from IPython.display import display, Image
res = pd.read_csv(ROOT / 'outputs' / 'a3_results_osm_anchored.csv')
print('Scenario rows:', len(res))
display(pd.read_csv(ROOT / 'outputs' / 'a3_verification_log.csv'))
display(res.pivot_table(index='variant', columns=['flow_proxy', 'w_m'], values='comfort_band_share_pct').round(1))
display(res.pivot_table(index='variant', columns=['flow_proxy', 'w_m'], values='corridor_overlap_m2').round(2))

In [ ]:
for name in ['osm_context.png', 'fig_six_placements.png', 'fig_base_comparison.png', 'fig_width_sensitivity.png']:
    display(Image(filename=str(ROOT / 'outputs' / name), width=820))

## 6. Change an assumption
Edit the `CONFIG` block near the top of `code/a3_osm_anchor.py` (window size, seat positions, w, c1, c2, s), then run sections 3–5 again. The reproduction check in section 4 will then report differences, which is expected. The known-case assertion (S1 = 80 %) holds only for the default CONFIG.